In [14]:
from bs4 import BeautifulSoup
import requests
from urllib.parse import urljoin, urldefrag
import networkx as nx

In [2]:
url = 'https://www.cdc.gov/foodborne-outbreaks/index.html'

In [3]:
page = requests.get(url)

In [4]:
soup = BeautifulSoup(page.text, 'html')

In [5]:
links = soup.find_all("a")

In [7]:
outbreak_links = set()
edges = []

for link in links:
    href = link.get('href')
    
    if href:
        full_url = urljoin(url, href)
        clean_url = urldefrag(full_url)[0]

        if 'cdc.gov/foodborne-outbreaks/' in clean_url:
            if clean_url != url:
                outbreak_links.add(clean_url)

for current_page in outbreak_links:
    edges.append((url, current_page))
    print(current_page)
                

https://www.cdc.gov/foodborne-outbreaks/outbreak-basics/investigation-timeline.html
https://www.cdc.gov/foodborne-outbreaks/outbreaks/index.html
https://www.cdc.gov/foodborne-outbreaks/outbreak-basics/roles-investigation.html
https://www.cdc.gov/foodborne-outbreaks/outbreak-basics/index.html
https://www.cdc.gov/foodborne-outbreaks/about/index.html
https://www.cdc.gov/foodborne-outbreaks/what-to-do/index.html
https://www.cdc.gov/foodborne-outbreaks/site.html
https://www.cdc.gov/foodborne-outbreaks/php/data-research/annual-summaries/index.html
https://www.cdc.gov/foodborne-outbreaks/php/confirming-cause/index.html
https://www.cdc.gov/foodborne-outbreaks/php/tools/index.html
https://www.cdc.gov/foodborne-outbreaks/php/rep-strains/index.html


In [8]:
for current_page in list(outbreak_links):
    current_url = requests.get(current_page)
    soup2 = BeautifulSoup(current_url.text, 'html')
    links2 = soup2.find_all('a')

    for new_link in links2:
        new_href = new_link.get('href')

        if new_href:
            full_url2 = urljoin(current_page, new_href)
            clean_url2 = urldefrag(full_url2)[0]
            
        if 'cdc.gov/foodborne-outbreaks/' in clean_url2:
            if clean_url2 != current_page:
                outbreak_links.add(clean_url2)
                edges.append((current_page, clean_url2))
                
print(edges)

[('https://www.cdc.gov/foodborne-outbreaks/index.html', 'https://www.cdc.gov/foodborne-outbreaks/outbreak-basics/investigation-timeline.html'), ('https://www.cdc.gov/foodborne-outbreaks/index.html', 'https://www.cdc.gov/foodborne-outbreaks/outbreaks/index.html'), ('https://www.cdc.gov/foodborne-outbreaks/index.html', 'https://www.cdc.gov/foodborne-outbreaks/outbreak-basics/roles-investigation.html'), ('https://www.cdc.gov/foodborne-outbreaks/index.html', 'https://www.cdc.gov/foodborne-outbreaks/outbreak-basics/index.html'), ('https://www.cdc.gov/foodborne-outbreaks/index.html', 'https://www.cdc.gov/foodborne-outbreaks/about/index.html'), ('https://www.cdc.gov/foodborne-outbreaks/index.html', 'https://www.cdc.gov/foodborne-outbreaks/what-to-do/index.html'), ('https://www.cdc.gov/foodborne-outbreaks/index.html', 'https://www.cdc.gov/foodborne-outbreaks/site.html'), ('https://www.cdc.gov/foodborne-outbreaks/index.html', 'https://www.cdc.gov/foodborne-outbreaks/php/data-research/annual-sum

In [9]:
G = nx.DiGraph()
G.add_edges_from(edges)

In [10]:
print(G.number_of_nodes())
print(G.number_of_edges())

27
136


In [31]:
degree = nx.degree_centrality(G)
top_degree = sorted(degree.items(), key = lambda x: x[1], reverse = True)
for page, score in top_degree[:5]:
    print(round(score,4), page)

1.2692 https://www.cdc.gov/foodborne-outbreaks/site.html
1.1538 https://www.cdc.gov/foodborne-outbreaks/php/data-research/annual-summaries/index.html
0.8462 https://www.cdc.gov/foodborne-outbreaks/index.html
0.8077 https://www.cdc.gov/foodborne-outbreaks/php/tools/index.html
0.7692 https://www.cdc.gov/foodborne-outbreaks/php/rep-strains/index.html


In [12]:
betweenness = nx.betweenness_centrality(G)
top_betweenness = sorted(betweenness.items(), key = lambda x: x[1], reverse = True)
for page, score in top_betweenness[:5]:
    print(round(score,4), page)

0.1354 https://www.cdc.gov/foodborne-outbreaks/site.html
0.1138 https://www.cdc.gov/foodborne-outbreaks/php/data-research/annual-summaries/index.html
0.0138 https://www.cdc.gov/foodborne-outbreaks/index.html
0.0077 https://www.cdc.gov/foodborne-outbreaks/php/tools/index.html
0.0 https://www.cdc.gov/foodborne-outbreaks/outbreak-basics/investigation-timeline.html


In [13]:
pagerank = nx.pagerank(G)
top_pagerank = sorted(pagerank.items(), key = lambda x: x[1], reverse = True)
for page, score in top_pagerank[:5]:
    print(round(score,4), page)

0.0699 https://www.cdc.gov/foodborne-outbreaks/site.html
0.0695 https://www.cdc.gov/foodborne-outbreaks/php/data-research/annual-summaries/index.html
0.0674 https://www.cdc.gov/foodborne-outbreaks/index.html
0.0669 https://www.cdc.gov/foodborne-outbreaks/php/tools/index.html
0.0663 https://www.cdc.gov/foodborne-outbreaks/outbreaks/index.html


In [35]:
nx.write_gexf(G, "cdc_foodbourne_network.gexf")